In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType ,IntegerType,DataType ,TimestampType,FloatType
catalog_name='ecommers'

# bronze_brands

In [0]:
df_bronze=spark.table(f'{catalog_name}.bronze.brz_brands')

In [0]:
df_silver=df_bronze.withColumn("brand_name",F.trim(F.col("brand_name")))

In [0]:
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), "[^a-zA-Z0-9]", ""))

In [0]:
# Anomalies dictionary
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

# PySpark replace is easy
df_silver = df_silver.replace(
    to_replace=anomalies,
    subset=["category_code"]
)

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_brands)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

# Category

In [0]:
df=spark.table(f"{catalog_name}.bronze.brz_category")

In [0]:
# Convert category_code to uppercase, trim spaces from category_code and category_name
df = df.withColumn("category_code", F.upper(F.trim(F.col("category_code")))) \
       .withColumn("category_name", F.trim(F.col("category_name")))

In [0]:
# Write cleaned data to the silver layer
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_category")

#Products

In [0]:
df=spark.table(f"{catalog_name}.bronze.brz_products")

In [0]:
# Convert category_code and brand_code to uppercase
df = df.withColumn("category_code", F.upper(F.col("category_code")))
df = df.withColumn("brand_code", F.upper(F.col("brand_code")))

# Drop rows where category_code is null
df = df.dropna(subset=["category_code"])

# Fix size column - trim spaces and standardize, and normalize to 'ONE SIZE'
df = df.withColumn("size", F.trim(F.col("size")))
df = df.withColumn("size", F.when(F.upper(F.col("size")) == "ONE-SIZE", "OS").otherwise(F.col("size")))

# Remove 'g' from weight_grams and cast to float
df = df.withColumn("weight_grams", F.regexp_replace(F.col("weight_grams"), "[gG]", "").cast(FloatType()))

In [0]:
# Fill null values in the 'color' column with 'Unknown'
df = df.withColumn("color", F.when(F.col("color").isNull(), "Unknown").otherwise(F.col("color")))

In [0]:
# Write cleaned data to the silver layer
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")

#products

In [0]:
df=spark.table(f"{catalog_name}.bronze.brz_customers")

In [0]:
# Remove rows where customer_id is null
df = df.dropna(subset=["customer_id"])

# Fill null values in string columns with 'Unknown'
string_cols = [c for c, dtype in df.dtypes if dtype == "string" and c != "customer_id"]
for col_name in string_cols:
    df = df.withColumn(col_name, F.when(F.col(col_name).isNull(), "Unknown").otherwise(F.col(col_name)))


# Calendar

In [0]:
df = spark.table(f"{catalog_name}.bronze.brz_calendar")

In [0]:
# Clean calendar data: uppercase day_name and ensure week_of_year is non-negative
df = df.withColumn("day_name", F.upper(F.col("day_name")))
df = df.withColumn("week_of_year", F.abs(F.col("week_of_year")))

# Validate week_of_year against the date column
df = df.withColumn("expected_week_of_year", F.weekofyear(F.col("date")))
df = df.withColumn("week_of_year_mismatch", F.col("week_of_year") != F.col("expected_week_of_year"))


In [0]:
# Remove duplicates from df
df = df.dropDuplicates()

In [0]:
# Write cleaned data to the silver layer
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")